In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ==============================================================================
# 1. SymPy Settings and Symbol Definitions
# ==============================================================================
sp.init_printing(use_unicode=True)
z_inv = sp.Symbol('z^{-1}', complex=True)
n = sp.Symbol('n', integer=True)

print("=== LTI System Analysis ===")

# 1. Impulse Response h[n] and Transfer Function H(z^-1)
h_n = (sp.Rational(1, 2))**n * sp.Heaviside(n)
H_z_inv = 1 / (1 - sp.Rational(1, 2) * z_inv)

print("\n1. Impulse response h[n] and Transfer Function H(z^-1):")
display(h_n)
display(H_z_inv)

# 2. Input Signal X(z^-1) and Output Z-Transform Y(z^-1)
X_z_inv = 1 / (1 - sp.Rational(1, 3) * z_inv) - 1 / (1 - 2 * z_inv)
X_z_inv = sp.simplify(X_z_inv)
Y_z_inv = sp.simplify(H_z_inv * X_z_inv)

print("\n2. Output Z-Transform Y(z^-1):")
display(Y_z_inv)

# 3. Partial Fraction Expansion of Y(z^-1)
pfe_y_inv = sp.apart(Y_z_inv, z_inv)

print("\n3. Partial Fraction Expansion of Y(z^-1):")
display(pfe_y_inv)

# 4. Final Output y[n]
y_n = (sp.Rational(10, 3) * (sp.Rational(1, 2))**n - 2 * (sp.Rational(1, 3))**n) * sp.Heaviside(n) + \
      sp.Rational(4, 3) * 2**n * sp.Heaviside(-n - 1)

print("\n4. Final analytical output y[n]:")
display(y_n)

# ==============================================================================
# 2. VISUALIZATION & INTERACTIVE PLOTS
# ==============================================================================
explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Analysis of System H(z)</b><br>
* <b>Frequency Response:</b> Magnitude (dB) and Phase (degrees) response.<br>
* <b>Pole-Zero Map:</b> Poles of H(z) indicated.<br>
* <b>Impulse Response:</b> Time-domain representation h[n].
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

def plot_system_analysis():
    with out:
        fig = plt.figure(figsize=(12, 11))
        gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.7])
        
        ax_mag = fig.add_subplot(gs[0, 0])
        ax_pz = fig.add_subplot(gs[0, 1])
        ax_phase = fig.add_subplot(gs[1, 0])
        ax_h = fig.add_subplot(gs[2, :])

        # A. Frequency Response - Magnitude
        omega = np.linspace(0, np.pi, 500)
        z_inv_val = np.exp(-1j * omega)
        H_omega = 1.0 / (1.0 - 0.5 * z_inv_val)
        
        mag_db = 20 * np.log10(np.abs(H_omega))
        phase_deg = np.angle(H_omega, deg=True)

        ax_mag.plot(omega / np.pi, mag_db, 'b')
        ax_mag.set_ylabel('Magnitude (dB)', color='b')
        ax_mag.grid(True)
        ax_mag.set_title('Frequency Response (Magnitude)', fontsize=10, fontweight='bold')

        # B. Pole-Zero Map (Moved to right column, next to magnitude)
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-1.5, 1.5)
        ax_pz.set_ylim(-1.5, 1.5)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)
        ax_pz.plot(np.cos(np.linspace(0, 2*np.pi, 100)), np.sin(np.linspace(0, 2*np.pi, 100)), 'k--', alpha=0.5)
        
        ax_pz.scatter([0.5], [0], s=140, color='red', marker='x', label='Pole at z=0.5')
        ax_pz.set_title('Pole-Zero Map of H(z)', fontsize=10, fontweight='bold')
        ax_pz.legend()

        # C. Frequency Response - Phase
        ax_phase.plot(omega / np.pi, phase_deg, 'r')
        ax_phase.set_ylabel('Phase (deg)', color='r')
        ax_phase.set_xlabel(r'Normalized Freq ($\pi$ rad/sample)')
        ax_phase.grid(True)
        ax_phase.set_title('Frequency Response (Phase)', fontsize=10, fontweight='bold')

        # D. Impulse Response h[n] (Reduced height)
        n_vec = np.arange(0, 15)
        h_vals = (0.5)**n_vec
        ax_h.stem(n_vec, h_vals, basefmt=" ")
        ax_h.set_title('Impulse Response h[n]', fontsize=10, fontweight='bold')
        ax_h.set_xlabel('n')
        ax_h.grid(True)

        plt.tight_layout()
        plt.show()

plot_system_analysis()
display(out)